# Bank Customer and Credit Risk Analysis with SQL

* Date: April 2026
* Tools: SQL (SQLite), Python, pandas
* Data: practice bank data, based on a public bank marketing dataset (not a real bank's records)

## Summary
I used SQL to answer the questions a bank's credit team and managers ask every month: who are the customers, which groups default most, how healthy is the loan book, and which marketing works.

### Key findings
- 9.9% of loans (586 of 5,906) have defaulted, worth KShs 686 million, or 9.1% of the loan book by value. At a 60% loss rate, that is about KShs 411 million in expected losses.
- Students default most (15.4%), but there are only 52 student borrowers, so this needs more data before acting on it. Self employed borrowers default least (7.1%).
- Bigger loans are not riskier. Default rates stay between 8% and 11% across every loan size.
- Savings accounts see more money withdrawn than deposited (about KShs 12.5 million net outflow), which is worth watching.
- Students and retired customers respond best to deposit campaigns (71% to 79% sign up when called on a mobile phone).

## The database

| Table | Rows | What it holds |
|-------|------|---------------|
| customers | 11,162 | Age, job, education, marital status |
| accounts | 11,162 | Account type, balance, credit card |
| loans | 5,906 | Loan type, amount, interest rate, status |
| transactions | 61,276 | Deposits, withdrawals, transfers |
| campaign | 11,162 | Marketing calls and whether the customer signed up |

All tables link through `customer_id`.

## 1. Setup and Database Connection

In [5]:
import sqlite3
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

conn = sqlite3.connect(r'C:\Users\siraa\Desktop\bank-sql-analysis\data\bank_database.db')

# Verify all tables loaded
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"Tables: {tables['name'].tolist()}")

# Quick row counts
for table in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as rows FROM {table}", conn)
    print(f"  {table}: {count['rows'][0]:,} rows")

Tables: ['customers', 'accounts', 'loans', 'transactions', 'campaign']
  customers: 11,162 rows
  accounts: 11,162 rows
  loans: 5,906 rows
  transactions: 61,276 rows
  campaign: 11,162 rows


## 2. Who Are the Customers?

Question: how do customers split by job and education, and which groups hold the most money? This helps decide which products to offer to whom.

In [6]:
# Customer breakdown by job and education
query = """
    SELECT c.job,
           c.education,
           COUNT(*) AS num_customers,
           ROUND(AVG(a.balance), 0) AS avg_balance,
           ROUND(AVG(a.balance) * COUNT(*), 0) AS total_segment_value
    FROM customers c
    JOIN accounts a ON c.customer_id = a.customer_id
    GROUP BY c.job, c.education
    HAVING COUNT(*) > 30
    ORDER BY avg_balance DESC
    LIMIT 15
"""

result = pd.read_sql(query, conn)
print("TOP 15 CUSTOMER SEGMENTS BY AVERAGE BALANCE")
print("=" * 65)
print(result.to_string(index=False))

TOP 15 CUSTOMER SEGMENTS BY AVERAGE BALANCE
          job education  num_customers  avg_balance  total_segment_value
   technician   unknown             52       2918.0             151760.0
      retired secondary            314       2730.0             857249.0
      retired  tertiary            140       2345.0             328241.0
self-employed  tertiary            230       2254.0             518431.0
      retired   primary            277       2224.0             616061.0
      unknown   unknown             39       2168.0              84550.0
 entrepreneur  tertiary            132       2090.0             275905.0
     services   unknown             41       2081.0              85322.0
    housemaid  tertiary             43       2069.0              88974.0
   management   unknown             84       2023.0             169943.0
      student  tertiary             83       2019.0             167554.0
   management   primary             66       1973.0             130227.0
   mana

What this shows: retired customers have some of the highest average balances (about 2,200 to 2,700). Managers with tertiary education are by far the largest group (2,178 customers), so they hold the most money in total even though their average balance is lower.

## 3. Which Customers Are the Biggest Credit Risk?

Question: which groups default most, and where should the credit team focus?

In [7]:
# Default rate by job category with loan exposure
query = """
    SELECT c.job,
           COUNT(DISTINCT c.customer_id) AS borrowers,
           COUNT(l.loan_id) AS total_loans,
           SUM(l.loan_amount) AS total_exposure,
           SUM(CASE WHEN l.loan_status = 'defaulted' THEN l.loan_amount ELSE 0 END) AS defaulted_amount,
           ROUND(AVG(CASE WHEN l.loan_status = 'defaulted' THEN 1.0 ELSE 0.0 END) * 100, 2) AS default_rate,
           ROUND(AVG(l.interest_rate), 2) AS avg_interest_rate
    FROM customers c
    JOIN loans l ON c.customer_id = l.customer_id
    GROUP BY c.job
    ORDER BY default_rate DESC
"""

result = pd.read_sql(query, conn)
print("CREDIT RISK BY CUSTOMER SEGMENT")
print("=" * 80)
print(result.to_string(index=False))

CREDIT RISK BY CUSTOMER SEGMENT
          job  borrowers  total_loans  total_exposure  defaulted_amount  default_rate  avg_interest_rate
      student         52           52        41250000           6400000         15.38              12.96
 entrepreneur        196          196       228650000          25150000         12.24              12.52
   technician        983          983      1270050000         129500000         10.68              12.57
      retired        155          155       211100000          31350000         10.32              12.42
   management       1202         1202      1480950000         137450000         10.32              12.47
     services        632          632       833400000          75800000         10.28              12.47
   unemployed        119          119       161100000          17000000         10.08              12.67
        admin        832          832      1009850000          84400000          9.25              12.67
  blue-collar       144

What this shows: students default most (15.4%), followed by entrepreneurs (12.2%). Self employed borrowers default least (7.1%). The student figure comes from only 52 loans, so one or two extra defaults would change it a lot.

In [8]:
# Default rate by loan size bucket: are bigger loans riskier?
query = """
    SELECT 
        CASE 
            WHEN loan_amount <= 100000 THEN '1. Under 100K'
            WHEN loan_amount <= 500000 THEN '2. 100K - 500K'
            WHEN loan_amount <= 1000000 THEN '3. 500K - 1M'
            WHEN loan_amount <= 2000000 THEN '4. 1M - 2M'
            ELSE '5. Over 2M'
        END AS loan_bucket,
        COUNT(*) AS num_loans,
        SUM(CASE WHEN loan_status = 'defaulted' THEN 1 ELSE 0 END) AS defaults,
        ROUND(AVG(CASE WHEN loan_status = 'defaulted' THEN 1.0 ELSE 0.0 END) * 100, 2) AS default_rate,
        ROUND(AVG(interest_rate), 2) AS avg_rate
    FROM loans
    GROUP BY loan_bucket
    ORDER BY loan_bucket
"""

result = pd.read_sql(query, conn)
print("DEFAULT RATE BY LOAN SIZE")
print("=" * 60)
print(result.to_string(index=False))

DEFAULT RATE BY LOAN SIZE
   loan_bucket  num_loans  defaults  default_rate  avg_rate
 1. Under 100K       1625       179         11.02     12.55
2. 100K - 500K       1704       155          9.10     12.46
  3. 500K - 1M        906        96         10.60     12.68
    4. 1M - 2M        816        85         10.42     12.57
    5. Over 2M        855        71          8.30     12.58


What this shows: bigger loans are not riskier here. Default rates stay between 8.3% and 11.0% across all loan sizes, and the largest loans (over 2 million) actually default least.

## 4. Where Does the Money Move?

Question: which account types are used most, and is money coming in or going out?

In [9]:
# Transaction volume and net flow by account type
query = """
    SELECT a.account_type,
           COUNT(t.transaction_id) AS total_transactions,
           ROUND(SUM(CASE WHEN t.transaction_type = 'deposit' THEN t.amount ELSE 0 END), 0) AS total_deposits,
           ROUND(SUM(CASE WHEN t.transaction_type = 'withdrawal' THEN t.amount ELSE 0 END), 0) AS total_withdrawals,
           ROUND(SUM(CASE WHEN t.transaction_type = 'deposit' THEN t.amount ELSE 0 END) - 
                 SUM(CASE WHEN t.transaction_type = 'withdrawal' THEN t.amount ELSE 0 END), 0) AS net_flow
    FROM accounts a
    JOIN transactions t ON a.account_id = t.account_id
    GROUP BY a.account_type
    ORDER BY total_transactions DESC
"""

result = pd.read_sql(query, conn)
print("TRANSACTION VOLUME BY ACCOUNT TYPE")
print("=" * 75)
print(result.to_string(index=False))

TRANSACTION VOLUME BY ACCOUNT TYPE
 account_type  total_transactions  total_deposits  total_withdrawals    net_flow
      savings               30921     183512079.0        196048803.0 -12536724.0
      current               21355     128796168.0        124604430.0   4191738.0
fixed_deposit                9000      52562535.0         57741791.0  -5179256.0


What this shows: savings accounts are used most, but more money left them than came in (about 12.5 million net outflow). Current accounts had a small net inflow. A steady outflow from savings can be an early sign that customers are moving their money elsewhere.

In [10]:
# Top 10 highest-value customers (total transaction volume)
query = """
    SELECT c.customer_id,
           c.job,
           c.education,
           a.balance AS current_balance,
           COUNT(t.transaction_id) AS num_transactions,
           ROUND(SUM(t.amount), 0) AS total_volume,
           RANK() OVER (ORDER BY SUM(t.amount) DESC) AS volume_rank
    FROM customers c
    JOIN accounts a ON c.customer_id = a.customer_id
    JOIN transactions t ON c.customer_id = t.customer_id
    GROUP BY c.customer_id, c.job, c.education, a.balance
    ORDER BY total_volume DESC
    LIMIT 10
"""

result = pd.read_sql(query, conn)
print("TOP 10 CUSTOMERS BY TRANSACTION VOLUME")
print("=" * 80)
print(result.to_string(index=False))

TOP 10 CUSTOMERS BY TRANSACTION VOLUME
 customer_id          job education  current_balance  num_transactions  total_volume  volume_rank
        8418 entrepreneur  tertiary              816                 8     3265999.0            1
        8181        admin secondary               67                 4     2582923.0            2
        4744      retired secondary             3738                 4     2381136.0            3
        1336     services secondary              522                 8     2278613.0            4
        3332   technician   unknown            14533                 8     2175659.0            5
        4644   management  tertiary              334                 7     2065577.0            6
        7775   technician secondary              788                 8     2040891.0            7
       11424     services secondary               21                 6     2021028.0            8
        7722  blue-collar secondary               94                 5     2009

What this shows: the most active customers are not always the richest. Several of the top ten by money moved hold balances under 1,000, so transaction activity and balance tell different stories.

## 5. How Healthy Is the Loan Book?

Question: how much of the loan book is performing, and how much money is at risk?

In [11]:
# Loan portfolio summary
query = """
    SELECT loan_status,
           COUNT(*) AS num_loans,
           ROUND(SUM(loan_amount), 0) AS total_amount,
           ROUND(AVG(loan_amount), 0) AS avg_loan,
           ROUND(AVG(interest_rate), 2) AS avg_rate,
           ROUND(SUM(loan_amount) * 1.0 / (SELECT SUM(loan_amount) FROM loans) * 100, 2) AS pct_of_portfolio
    FROM loans
    GROUP BY loan_status
    ORDER BY total_amount DESC
"""

result = pd.read_sql(query, conn)
print("LOAN PORTFOLIO BREAKDOWN")
print("=" * 75)
print(result.to_string(index=False))

LOAN PORTFOLIO BREAKDOWN
loan_status  num_loans  total_amount  avg_loan  avg_rate  pct_of_portfolio
     active       3910  5067950000.0 1296151.0     12.53             67.39
       paid       1410  1766750000.0 1253014.0     12.57             23.49
  defaulted        586   685750000.0 1170222.0     12.64              9.12


What this shows: 67% of the loan book by value is active, 23% is fully repaid and 9.1% has defaulted.

In [12]:
# Estimated loss from defaults (assuming 60% LGD)
query = """
    SELECT 
        COUNT(*) AS defaulted_loans,
        SUM(loan_amount) AS total_defaulted_amount,
        ROUND(SUM(loan_amount) * 0.60, 0) AS estimated_loss_60pct_lgd,
        ROUND(AVG(loan_amount), 0) AS avg_defaulted_loan,
        ROUND(SUM(loan_amount) * 1.0 / (SELECT SUM(loan_amount) FROM loans) * 100, 2) AS pct_of_portfolio
    FROM loans
    WHERE loan_status = 'defaulted'
"""

result = pd.read_sql(query, conn)
print("DEFAULT LOSS ESTIMATION (60% LGD)")
print("=" * 60)
print(result.to_string(index=False))

DEFAULT LOSS ESTIMATION (60% LGD)
 defaulted_loans  total_defaulted_amount  estimated_loss_60pct_lgd  avg_defaulted_loan  pct_of_portfolio
             586               685750000               411450000.0           1170222.0              9.12


What this shows: if the bank recovers 40% of each defaulted loan (a 60% loss rate, a common assumption), it would lose about 411 million. This is the kind of figure a bank sets aside as a provision.

## 6. Which Marketing Works Best?

Question: which customers sign up for a deposit account when contacted, and does the way I contact them matter?

In [13]:
# Campaign conversion rate by job and contact method
query = """
    SELECT c.job,
           camp.contact_method,
           COUNT(*) AS contacts,
           SUM(CASE WHEN camp.subscribed_deposit = 'yes' THEN 1 ELSE 0 END) AS conversions,
           ROUND(AVG(CASE WHEN camp.subscribed_deposit = 'yes' THEN 1.0 ELSE 0.0 END) * 100, 2) AS conversion_rate,
           ROUND(AVG(camp.call_duration_sec), 0) AS avg_call_duration
    FROM customers c
    JOIN campaign camp ON c.customer_id = camp.customer_id
    GROUP BY c.job, camp.contact_method
    HAVING COUNT(*) > 50
    ORDER BY conversion_rate DESC
    LIMIT 15
"""

result = pd.read_sql(query, conn)
print("CAMPAIGN CONVERSION BY SEGMENT & CHANNEL")
print("=" * 75)
print(result.to_string(index=False))

CAMPAIGN CONVERSION BY SEGMENT & CHANNEL
          job contact_method  contacts  conversions  conversion_rate  avg_call_duration
      student       cellular       311          246            79.10              337.0
      retired      telephone       170          129            75.88              439.0
      retired       cellular       518          366            70.66              374.0
   unemployed       cellular       275          173            62.91              444.0
   management       cellular      2066         1154            55.86              361.0
        admin       cellular       972          529            54.42              344.0
self-employed       cellular       287          153            53.31              385.0
   technician       cellular      1392          731            52.51              373.0
        admin      telephone        84           41            48.81              406.0
     services       cellular       627          303            48.33           

What this shows: students (79%) and retired customers (71% to 76%) sign up most often. Calling mobile phones generally works better than landlines for the same type of customer, for example 52.5% against 34.2% for technicians.

## 7. One View of Each Customer Group

Question: can I see balances, loans, defaults and activity for each group in one table?

In [14]:
# Full customer segment dashboard
query = """
    SELECT c.job,
           COUNT(DISTINCT c.customer_id) AS customers,
           ROUND(AVG(a.balance), 0) AS avg_balance,
           ROUND(AVG(l.loan_amount), 0) AS avg_loan,
           ROUND(AVG(l.loan_amount) * 1.0 / NULLIF(AVG(a.balance), 0), 1) AS loan_to_balance,
           ROUND(AVG(CASE WHEN l.loan_status = 'defaulted' THEN 1.0 ELSE 0.0 END) * 100, 2) AS default_rate,
           ROUND(AVG(l.interest_rate), 2) AS avg_rate,
           (SELECT ROUND(AVG(sub_t.txn_count), 1) 
            FROM (SELECT customer_id, COUNT(*) as txn_count 
                  FROM transactions GROUP BY customer_id) sub_t
            WHERE sub_t.customer_id IN (
                SELECT customer_id FROM customers WHERE job = c.job
            )) AS avg_transactions
    FROM customers c
    JOIN accounts a ON c.customer_id = a.customer_id
    JOIN loans l ON c.customer_id = l.customer_id
    GROUP BY c.job
    HAVING COUNT(DISTINCT c.customer_id) > 50
    ORDER BY default_rate DESC
"""

result = pd.read_sql(query, conn)
print("CUSTOMER RISK DASHBOARD")
print("=" * 95)
print(result.to_string(index=False))
print("\nKey: loan_to_balance = how leveraged the segment is (higher = more risk)")

CUSTOMER RISK DASHBOARD
          job  customers  avg_balance  avg_loan  loan_to_balance  default_rate  avg_rate  avg_transactions
      student         52       1585.0  793269.0            500.5         15.38     12.96               5.4
 entrepreneur        196       1707.0 1166582.0            683.5         12.24     12.52               5.7
   technician        983       1377.0 1292014.0            938.2         10.68     12.57               5.5
      retired        155       1779.0 1361935.0            765.7         10.32     12.42               5.5
   management       1202       1584.0 1232072.0            777.7         10.32     12.47               5.5
     services        632        971.0 1318671.0           1358.6         10.28     12.47               5.4
   unemployed        119       1028.0 1353782.0           1317.3         10.08     12.67               5.4
        admin        832        902.0 1213762.0           1345.2          9.25     12.67               5.5
  blue-collar

What this shows: average loans are hundreds of times larger than average account balances in every group. This compares two different things (a loan amount against a current balance), so it is a rough sign of how stretched borrowers are, not an exact debt ratio.

## 8. Key Findings and Recommendations

### Credit risk
- Students have the highest default rate (15.4%), but from only 52 loans. Worth watching, and the bank could ask for a guarantor on student loans.
- Loan size does not change the default rate much, so loan limits alone will not reduce defaults.

### Customers and accounts
- Savings accounts had a net outflow of about 12.5 million. The bank should check whether customers are moving money to competitors.
- Retired customers hold high balances and respond well to campaigns, so they are a good target for savings and investment products.

### Marketing
- Students and retired customers sign up most often.
- Mobile phone calls convert better than landline calls for the same customer groups.

### Loan book
- 9.9% of loans by number (9.1% by value) have defaulted.
- Expected losses of about 411 million at a 60% loss rate show how much the bank may need to set aside.

### Limits
- This is practice data, not a real bank's records.
- Some groups are small (for example 52 students), so their rates can swing a lot.
- No dates are used, so the analysis cannot show whether things are getting better or worse over time.